# Sample 06: 高度な信頼性・エンコーディング機能 (v0.2.0+)

`binary_master` に備わる高信頼プロトコルモデリングと最新エンコーディング機能の総合チュートリアルです。

### 学べる内容
- `CRC32` 宣言による自動チェックサム計算と改ざん検知
- `BinaryEnum[Type]` による型安全な Python Enum マッピング
- `Magic[b'...']` によるファイルシグネチャ自動検証
- `Constant[Type, Value]` によるプロトコルバージョン固定値強制
- `.to_json()` / `.from_json()` による JSON / 辞書相互変換
- `reader.iter_struct()` による大容量ファイルのゼロコピーストリーミング
- `BitWriter` / `BitReader` による任意ビット幅ストリームの直列パッキング

In [1]:
from binary_master import (
    CRC32,
    BinaryEnum,
    BinaryReader,
    Constant,
    Endian,
    FixedString,
    Magic,
    UInt8,
    UInt16,
    VarInt,
    VarUInt,
    binary_struct,
    hexdump,
)
from binary_master.bitstream import BitReader, BitWriter

## 1. 高度なパケット構造体の定義

`Magic` は初期値が自動設定されコンストラクタ引数から除外されます。`CRC32` は直前バイトまでの巡回冗長検査値をシリアライズ時に自動計算し、読み込み時に自動検証します。

In [2]:
class MessageType(BinaryEnum):
    HEARTBEAT = 0x01
    DATA_PAYLOAD = 0x02
    SHUTDOWN = 0xFF

@binary_struct(endian="big")
class AdvancedPacket:
    """Magic, Constant, Enum, VarInt, CRC32 を統合した高信頼パケット."""
    magic: Magic[b"PKT\x01"]              # 自動で b'PKT\x01' が入り、読み込み時に不一致なら即座に例外
    version: Constant[UInt16, 2]         # バージョン固定値 2
    msg_type: MessageType[UInt8]         # Enum を 1 バイト (UInt8) としてシリアライズ
    sequence: VarUInt                    # LEB128 可変長符号なし整数
    temperature_delta: VarInt            # LEB128 可変長符号付き整数
    label: FixedString[8]                # 8バイト固定文字列
    checksum: CRC32                      # 自動計算・検証される IEEE 802.3 CRC32

# magic と version は引数不要！
packet = AdvancedPacket(
    msg_type=MessageType.DATA_PAYLOAD,
    sequence=1024,
    temperature_delta=-5,
    label="SENSOR_A",
)

raw = packet.to_bytes()
print(f"シリアライズバイナリ ({len(raw)} バイト):")
print(hexdump(raw, annotate=True))

restored = AdvancedPacket.from_bytes(raw)
print(f"Magic:       {restored.magic}")
print(f"Version:     {restored.version}")
print(f"Msg Type:    {restored.msg_type.name} ({restored.msg_type.value})")
print(f"Sequence:    {restored.sequence}")
print(f"CRC32:       0x{restored.checksum:08X}")
assert restored.checksum != 0

シリアライズバイナリ (22 バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  50 4b 54 01 00 02 02 80  08 7b 53 45 4e 53 4f 52  |PKT......{SENSOR|
00000010  5f 41 33 01 3c b3                                 |_A3.<.          |
  [Total: 22 bytes (`0x0016`)]
Magic:       b'PKT\x01'
Version:     2
Msg Type:    DATA_PAYLOAD (2)
Sequence:    1024
CRC32:       0x33013CB3

## 2. JSON / 辞書相互変換 (`to_json`, `from_json`)

In [3]:
json_str = packet.to_json(indent=2, bytes_format="hex")
print("JSON 表現:\n", json_str)

from_json_obj = AdvancedPacket.from_json(json_str)
assert from_json_obj.sequence == packet.sequence
assert from_json_obj.label == packet.label
print("JSON からの復元成功！")

JSON 表現:
 {
  "magic": "0x504b5401",
  "version": 2,
  "msg_type": "DATA_PAYLOAD",
  "sequence": 1024,
  "temperature_delta": -5,
  "label": "SENSOR_A",
  "checksum": 855719091
}
JSON からの復元成功！

## 3. 大容量ファイルのストリーミングイテレーション (`iter_struct`)

全データをリスト化せずイテレータとして1件ずつ復元するため、メモリ消費を最小限に抑えられます。

In [4]:
packets = [
    AdvancedPacket(msg_type=MessageType.HEARTBEAT, sequence=i, temperature_delta=i - 2, label=f"DEV_{i}")
    for i in range(3)
]
stream_bytes = b"".join(p.to_bytes() for p in packets)

reader = BinaryReader(stream_bytes, default_endian=Endian.BIG)
print("ストリーミング読み込み結果:")
for idx, p in enumerate(reader.iter_struct(AdvancedPacket)):
    print(f"  Packet {idx}: seq={p.sequence}, label={p.label!r}, crc=0x{p.checksum:08X}")

ストリーミング読み込み結果:
  Packet 0: seq=0, label='DEV_0', crc=0x503834D4
  Packet 1: seq=1, label='DEV_1', crc=0x103D2CCC
  Packet 2: seq=2, label='DEV_2', crc=0xC6A53425

## 4. 任意ビット幅ストリームの直列パッキング (`BitWriter`, `BitReader`)

バイト境界に揃わない任意ビット数（3bit, 5bit, 12bit 等）を連続して詰め込み・復元できます。

In [5]:
bw = BitWriter()
bw.write_bits(0b1101, 4)      # 4 bits
bw.write_bits(0b010, 3)       # 3 bits
bw.write_bits(0b1, 1)         # 1 bit -> byte 0: 0b11010101 (0xD5)
bw.write_bits(0b11110000, 8)  # 8 bits -> byte 1: 0xF0
bit_data = bw.to_bytes()

print(f"{bw.total_bits} ビットを {len(bit_data)} バイトにパッキング: 0x{bit_data.hex()}")

br = BitReader(bit_data)
b1 = br.read_bits(4)
b2 = br.read_bits(3)
b3 = br.read_bits(1)
b4 = br.read_bits(8)
print(f"復元結果: 4bit={bin(b1)}, 3bit={bin(b2)}, 1bit={bin(b3)}, 8bit={bin(b4)}")

assert b1 == 0b1101 and b4 == 0b11110000
print("高度機能の全検証成功！")

16 ビットを 2 バイトにパッキング: 0xd5f0
復元結果: 4bit=0b1101, 3bit=0b10, 1bit=0b1, 8bit=0b11110000
高度機能の全検証成功！